In [68]:
import pandas as pd
import requests
import os
from calculadora_inflacao import CalculadoraFatorInflacao

# APOGESP
## Cálculo de impacto orçamentário das propostas reajuste salarial para os APPGGS

Esse notebook realiza a estimativa de impacto orçamentário das propostas de reajuste salarial feitas pela APOGESP para a campanha salarial de 2026 na Prefeitura de São Paulo.

São consideradas duas propostas principais:
1. Reajuste inflacionário, que se subdivide em:
    1. Reajuste da tabela original da carreira considerando a inflação acumulada desde as primeiras nomeações (jun/2026)
    2. Reajuste da tabela da carreira considerando a inflação acumulada na gestão Ricardo Nunes e a tabela no momento de início da gestão (jun/2021)
2. Equiparação com a tabela dos AMCI (ou seja, unificação da estrutura remuneratória do quadro QPPGG)


Em ambos os casos, a estimativa segue a seguinte metodologia:

1. Identificação dos APPGGs em exercício (incluso cedidos) segundo os últimos dados disponíveis no portal de dados abertos (dezembro de 2025),
2. Atualização dos dados do item 1 considerando o acréscimo dos 30 APPGGs recém-nomeados no nível 1, criando dados sintéticos (novas linhas na tabela)
3. Cálculo do salário base atual de cada APPGG considerando a tabela atual e o nível em que ele se encontra,
4. Cálculo dos encargos e benefícios que tem como base o salário atual como contribuição previdenciária obrigatória, auxílio transporte, auxílio alimentação etc.
5. Cálculo do salário proposto considerando o nível em que o APPGG se encontra atualmente e a tabela atualizada proposta
6. Cálculo dos encargos citados no item 6,
7. Somatória dos salários e encargos para cada APPGG considerando a tabela atual e a tabela proposta,
8. Cálculo da diferença entre a situação atual e a situação proposta conforme item 7,
9. Somatória da diferença por nível da carreira APPGG (p. ex., para o nível 1 a proposta implica em X mil reais mensais para a Prefeitura para o nível 2, y mil reais etc.)
10. Simulação de progressão da carreira considerando 18 meses para a "subida" de nível e projeção do impacto orçamentário anual até o final da gestão (2028)

O diagrama na imagem abaixo representa as etapas da metodologia.

![Grafo da metodologia](mermaid_graph.svg)

## Dados servidores

Nessa seção pegamos os dados dos servidores ativos mais atualizados (dezembro de 2025) do portal de dados abertos. Em seguida, identificamos os APPGGs e adicionamos dados sintéticos referentes aos APPGGs recém-nomeados.

In [69]:
URL_SERVIDORES = 'https://dados.prefeitura.sp.gov.br/dataset/bf5df0f4-4fb0-4a5e-b013-07d098cc7b1c/resource/e4c65839-3bc8-4035-b0a7-108ec8740536/download/verificado_ativos_05-01-2026_dez-2025in.csv'
file_dados = 'servidores_ativos.csv'

In [70]:
def download_dados_servidores(fname:str, url:str=URL_SERVIDORES)->str:

    if not os.path.exists(fname):
        print('Downloading data')
        with requests.get(url) as r:
            r.raise_for_status()
            content = r.content
            with open(fname, 'wb') as f:
                f.write(content)
            
            assert os.path.exists(fname)
            return fname
    return fname

In [71]:
file_dados = download_dados_servidores(file_dados)

In [72]:
df = pd.read_csv(file_dados, encoding='latin1', sep=';')

In [73]:
df.head()

,REGISTRO,VINCULO,NOME,CARGO_BASICO,REF_CARGO_BAS,SEGMENTO,GRUPO,SUBGRUPO,ESCOL_CARGO_BASICO,CARGO_COMISSAO,...,SECRET_SUBPREF,SIGLA,SETOR,DISTRITO,SUBPREFEITURA,ORGAO_EXT,SEXO,ANO_NASCIMENTO,RACA_COR,PCD
0,1145541,16,CLEUZA BORGES PEREIRA SILVA,ASSESSOR IV,CDA-4,NaN,QC,CARGO EM COMISSAO,NAO SE APLICA,NaN,...,SECRETARIA MUNICIPAL DE GESTAO,SEGES,ASSESSORIA JURIDICA,SE,SE,NaN,FEMININO,1949,PARDA,NAO
1,1160206,5,ADILSON DA SILVA,COORDENADOR II,CDA-6,NaN,QC,CARGO EM COMISSAO,SUPERIOR COMPLETO,NaN,...,SUBPREFEITURA SANTO AMARO,SUB-SA,COORDENADORIA DE PLANEJAMENTO E DESENVOLVIMENT...,SANTO AMARO,SANTO AMARO,NaN,MASCULINO,1949,BRANCA,NAO
2,1160478,2,JULIO DE CARVALHO,FISCAL DE POSTURAS MUNICIPAIS NIVEL IV,QFPM15,NaN,QFPM,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SUBPREFEITURA SANTANA/TUCURUVI,SUB-ST,SUPERVISÃO TECNICA DE FISCALIZACAO,TUCURUVI,SANTANA/TUCURUVI,NaN,MASCULINO,1951,BRANCA,NAO
3,1161181,9,NEUSA PEDRAO NASSIR,ASSESSOR V,CDA-5,NaN,QC,CARGO EM COMISSAO,NAO SE APLICA,NaN,...,SECRETARIA DO GOVERNO MUNICIPAL,SGM,ASSESSORIA JURIDICA,SE,SE,NaN,FEMININO,1947,BRANCA,NAO
4,1168185,4,MARIA HELENA DI VERNIERI CUPPARI,ASSISTENTE TECNICO EDUCACIONAL,QPE17A,NaN,QPE L. 14660/07 RGPS,CARGO EM COMISSAO,LICENCIATURA PLENA COMPLETA,NaN,...,SECRETARIA MUNICIPAL DE EDUCACAO,SME,DIRETORIA REGIONAL DE EDUCACAO PIRITUBA/JARAGUA,LAPA,LAPA,NaN,FEMININO,1942,BRANCA,NAO


In [74]:
cargos = df['REF_CARGO_BAS'].unique()

In [75]:
for cargo in cargos:
    if "appgg" in str(cargo).lower():
        print(cargo)

APPGG2
APPGG1
APPGG5
APPGG6
APPGG4
APPGG3


In [76]:
appggs = df[df['REF_CARGO_BAS'].str.contains('APPGG')].reset_index(drop=True)

In [77]:
appggs.head()

,REGISTRO,VINCULO,NOME,CARGO_BASICO,REF_CARGO_BAS,SEGMENTO,GRUPO,SUBGRUPO,ESCOL_CARGO_BASICO,CARGO_COMISSAO,...,SECRET_SUBPREF,SIGLA,SETOR,DISTRITO,SUBPREFEITURA,ORGAO_EXT,SEXO,ANO_NASCIMENTO,RACA_COR,PCD
0,6394604,3,CLAUDIO AGUIAR ALMEIDA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG2,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE CULTURA E ECONOMIA CRI...,SMC,SECRETARIA MUNICIPAL DE CULTURA E ECONOMIA CRI...,SE,SE,NaN,MASCULINO,1963,PRETA,NAO
1,7575491,2,MAURICIO DA SILVA CORREIA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG1,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE GESTAO,SEGES,SECRETARIA MUNICIPAL DE GESTAO,SE,SE,PREFEITURA DO MUNICIPIO DE ITAJAI,MASCULINO,1986,BRANCA,NAO
2,7718543,6,MARCIA MIYUKI ISHIKAWA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG2,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE HABITACAO,SEHAB,DEPARTAMENTO DE PLANEJAMENTO HABITACIONAL,SE,SE,NaN,FEMININO,1978,BRANCA,NAO
3,7794720,2,TIAGO ROSA MACHADO,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG2,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,ASSESSOR III,...,SECRETARIA MUNICIPAL DE ESPORTES E LAZER,SEME,SECRETARIA MUNICIPAL DE ESPORTES E LAZER,MOEMA,VILA MARIANA,NaN,MASCULINO,1983,BRANCA,NAO
4,7840501,2,THAIS ROBERTO DA SILVA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG5,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE DIREITOS HUMANOS E CID...,SMDHC,SECRETARIA MUNICIPAL DE DIREITOS HUMANOS E CID...,SE,SE,NaN,FEMININO,1977,PRETA,NAO


In [78]:
appggs = appggs[['REGISTRO', 'NOME', 'REF_CARGO_BAS', 'SIGLA', 'DATA_INICIO_EXERC']]

renomear_cols = {
    'REGISTRO' : 'rf',
    'NOME' : 'nome',
    'REF_CARGO_BAS' : 'cargo_base',
    'SIGLA' : 'secretaria_dez_2025',
    'DATA_INICIO_EXERC' : 'dt_inicio_exercicio'
}

appggs.rename(renomear_cols, axis=1, inplace=True)


In [79]:
appggs['nivel_carreira'] = appggs['cargo_base'].str.extract(r'(\d+)$').astype(int)

In [80]:
appggs.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,01/10/2021,2
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,03/11/2021,1
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,08/12/2021,2
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,05/01/2022,2
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,29/11/2017,5


In [81]:
appggs.shape

(170, 6)

In [82]:
recem_nomeados = [{
            'rf' : str(i+1).zfill(7),
            'nome' : f'recem_nomeado_{i+1}',
            'cargo_base' : 'APPGG1',
            'secretaria_dez_2025' : 'NA',
            'nivel_carreira' : 1,
            'dt_inicio_exercicio' : '01/03/2026'
            }
            for i in range(30)
            ]

recem_nomeados = pd.DataFrame(recem_nomeados)

In [83]:
recem_nomeados.head()

,rf,nome,cargo_base,secretaria_dez_2025,nivel_carreira,dt_inicio_exercicio
0,0000001,recem_nomeado_1,APPGG1,NA,1,01/03/2026
1,0000002,recem_nomeado_2,APPGG1,NA,1,01/03/2026
2,0000003,recem_nomeado_3,APPGG1,NA,1,01/03/2026
3,0000004,recem_nomeado_4,APPGG1,NA,1,01/03/2026
4,0000005,recem_nomeado_5,APPGG1,NA,1,01/03/2026


In [84]:
recem_nomeados.shape

(30, 6)

In [85]:
appggs = pd.concat([appggs, recem_nomeados])

In [86]:
appggs.shape

(200, 6)

# Situaçao atual

Nessa seção calculamos a situação atual de dispêndio com a carreira de APPGGs por parte da Prefeitura, considerando apenas os vencimentos e os encargos que estão relacionados aos vencimentos, como contribuição previdenciaria.

In [87]:
tabela_atual = {
    1 : 13208.14,
    2 : 14528.96,
    3 : 14892.18,
    4 : 15264.48,
    5 : 15646.09,
    6 : 16037.24,
    7 : 17961.73,
    8 : 18410.77,
    9 : 18871.04,
    10 : 19342.80,
    11 : 19826.39,
    12 : 21869.74,
    13 : 22416.47,
    14 : 22976.89,
    15 : 23551.31
}

In [88]:
appggs['vencimento'] = appggs['nivel_carreira'].map(tabela_atual)

In [89]:
#vou colocar como datetime porque precisamos calcular com base na data
appggs['dt_inicio_exercicio'] = pd.to_datetime(appggs['dt_inicio_exercicio'], format ="%d/%m/%Y")


#inclusive ja vou definir se a pessoa contribui para o regime proprio (IPREM) ou nao
appggs['contribui_rpps'] = appggs['dt_inicio_exercicio'].dt.year<2018

In [90]:
appggs.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,vencimento,contribui_rpps
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,14528.96,False
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,13208.14,False
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,14528.96,False
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,14528.96,False
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,15646.09,True


### Terço adicional de férias e décimo terceiro

Nesse caso vamos dividir ambos por 12 são gastos anuais e nossa base é mensal.

In [91]:
appggs_atual = appggs.copy(deep=True)

In [92]:
def calcular_decimo_terceiro(row):

    return round(row['vencimento']/12, 2)

def calcular_terco_ferias(row):

    return round((row['vencimento']/3)/12, 2)

In [93]:
appggs_atual['decimo_terceiro'] = appggs_atual.apply(calcular_decimo_terceiro, axis=1)

appggs_atual['terco_ferias'] = appggs_atual.apply(calcular_terco_ferias, axis=1)

In [94]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,vencimento,contribui_rpps,decimo_terceiro,terco_ferias
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,14528.96,False,1210.75,403.58
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,13208.14,False,1100.68,366.89
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,14528.96,False,1210.75,403.58
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,14528.96,False,1210.75,403.58
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,15646.09,True,1303.84,434.61


### Vale alimentação

O vale alimentação só é devido para quem ganha menos de 10 salários minimos. Como com o reajuste essa proporcao pode diminuir, ele deve entrar na conta apesar de ser um valor fixo.

In [95]:
def calcular_va(row):
    SALARIO_MINIMO = 1621
    VALOR_VALE_ALIMENTACAO = 325.25

    if row['vencimento'] <= SALARIO_MINIMO*10:
        return VALOR_VALE_ALIMENTACAO
    return 0

In [96]:
appggs_atual['vale_alimentacao'] = appggs_atual.apply(calcular_va, axis=1)

In [97]:
appggs_atual.sample(3)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,vencimento,contribui_rpps,decimo_terceiro,terco_ferias,vale_alimentacao
161,9412000,ODILON ALVES CANDIDO,APPGG1,PGM,2024-07-19,1,13208.14,False,1100.68,366.89,325.25
41,8359091,DANIEL BRUNO GARCIA,APPGG6,SEPLAN,2016-06-22,6,16037.24,True,1336.44,445.48,325.25
122,8915296,ALEXANDRE ALONSO DURANTE,APPGG2,SF,2022-02-01,2,14528.96,False,1210.75,403.58,325.25


### Contribuicao previdenciaria

Para calcular a contribuicao previdenciaria precisamos saber se a pessoa entrou antes de 2018 (ou seja, se é do IPREM) ou depois (se é do SAMPAPREV)

In [ ]:
def valor_iprem(row):

    CONTRIBUICAO_IPREM = 0.28

    if not row['contribui_rpps']:
        return 0
    
    # o iprem nao considera o terço adicional de férias
    valor_base = row['vencimento'] + row['decimo_terceiro']
    
    return round(valor_base * CONTRIBUICAO_IPREM, 2)

In [99]:
appggs_atual['contribuicao_iprem'] = appggs_atual.apply(valor_iprem, axis=1)

In [100]:
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,vencimento,contribui_rpps,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem
15,0000016,recem_nomeado_16,APPGG1,NA,2026-03-01,1,13208.14,False,1100.68,366.89,325.25,0.0
148,9411836,LAURA MENEGHEL DOS SANTOS,APPGG1,SEME,2024-08-02,1,13208.14,False,1100.68,366.89,325.25,0.0
141,9411747,GIANLUCCA VERGIAN DALENOGARE,APPGG1,SMS,2024-07-19,1,13208.14,False,1100.68,366.89,325.25,0.0
13,8116377,DIOGO FRIZZO DE MEDEIROS,APPGG1,SEGES,2024-07-23,1,13208.14,False,1100.68,366.89,325.25,0.0


In [103]:
def contribuicao_inss(row):

    ALIQUOTA_INSS = 0.21
    TETO_INSS = 8475.55

    if row['contribui_rpps']:
        return 0
    
    # o inss vai sobre todos os vencimentos, incluso terco adicional
    valor_base = row['vencimento'] + row['terco_ferias'] + row['decimo_terceiro']

    #só para validar o teto - no caso de APPGG da na mesma porque tá todo mundo acima, mas pra deixar a funcao correta
    if valor_base > TETO_INSS:
        valor_base = TETO_INSS
        
    return round(valor_base*ALIQUOTA_INSS, 2)
    


In [104]:
appggs_atual['contribuicao_inss'] = appggs_atual.apply(contribuicao_inss, axis=1)

In [105]:
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,vencimento,contribui_rpps,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem,contribuicao_inss
36,8359016,SANDRO LUIS PALANCA,APPGG5,SEPLAN,2016-07-04,5,15646.09,True,1303.84,434.61,325.25,3559.49,0.00
19,8231605,JOSE RADAMES MARQUES MIGUEL DOS ANJOS,APPGG2,SEGES,2022-01-17,2,14528.96,False,1210.75,403.58,325.25,0.00,1779.87
72,8414599,LOUISE RODRIGUES DE VASCONCELOS CORACINI,APPGG4,SMDET,2017-06-09,4,15264.48,True,1272.04,424.01,325.25,3472.67,0.00
103,8897042,MARY KAWAUCHI,APPGG2,SEGES,2021-10-29,2,14528.96,False,1210.75,403.58,325.25,0.00,1779.87


In [106]:
def previdencia_complementar(row):

    TETO_INSS = 8475.55
    ALIQUOTA_COMPLEMENTAR = 0.075
    if row['contribui_rpps']:
        return 0
    
    valor_base = row['vencimento'] + row['decimo_terceiro']
    #só contribui o que é acima do teto
    valor_base = valor_base - TETO_INSS

    return round(valor_base * ALIQUOTA_COMPLEMENTAR, 2)



In [108]:
appggs_atual['previdencia_complementar'] = appggs_atual.apply(previdencia_complementar, axis=1)

In [109]:
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,vencimento,contribui_rpps,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem,contribuicao_inss,previdencia_complementar
113,8907536,KIM FERREIRA DE SOUZA,APPGG1,SEGES,2021-12-17,1,13208.14,False,1100.68,366.89,325.25,0.00,1779.87,437.5
3,0000004,recem_nomeado_4,APPGG1,NA,2026-03-01,1,13208.14,False,1100.68,366.89,325.25,0.00,1779.87,437.5
13,0000014,recem_nomeado_14,APPGG1,NA,2026-03-01,1,13208.14,False,1100.68,366.89,325.25,0.00,1779.87,437.5
53,8359431,FERNANDA BRAZ TOBIAS DE AGUIAR,APPGG6,SMS,2016-06-21,6,16037.24,True,1336.44,445.48,325.25,3648.47,0.00,0.0


## Valor total

Agora calculamos o valor total e salvamos os dados

In [112]:
def valor_total_prefeitura(row):

    vencimentos = row['vencimento'] + row['decimo_terceiro'] + row['terco_ferias']
    auxilios = row['vale_alimentacao']
    previdencia = row['contribuicao_iprem'] + row['contribuicao_inss'] + row['previdencia_complementar']

    return vencimentos+auxilios+previdencia

In [113]:
appggs_atual['valor_total_prefeitura'] = appggs_atual.apply(valor_total_prefeitura, axis=1)

In [114]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,vencimento,contribui_rpps,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,valor_total_prefeitura
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,14528.96,False,1210.75,403.58,325.25,0.00,1779.87,544.81,18793.22
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,13208.14,False,1100.68,366.89,325.25,0.00,1779.87,437.50,17218.33
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,14528.96,False,1210.75,403.58,325.25,0.00,1779.87,544.81,18793.22
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,14528.96,False,1210.75,403.58,325.25,0.00,1779.87,544.81,18793.22
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,15646.09,True,1303.84,434.61,325.25,3559.49,0.00,0.00,21269.28


In [115]:
appggs_atual.to_csv('situacao_atual.csv', sep=';')